# 05 · Query-based detector: RT-DETR

RT-DETR (Ultralytics, `rtdetr-l`) encodes multi-scale CNN features with a hybrid encoder, selects the top-`num_queries` encoder tokens as initial queries, and refines them with a 6-layer decoder. Each query predicts one box or "no object". Training matches predictions to ground truth with the **Hungarian algorithm** (one-to-one, cost = class + L1 + GIoU), so duplicates are penalised by the loss itself and **no NMS** is needed at inference.

Budget: same frames, `imgsz` and number of epochs as the CNN detectors; GPU time is reported.

Experiments: **D0** 300 queries → **D1** queries scaled to scene density → **D2** matching cost with a larger GIoU weight → **D3** without denoising queries (to see their effect on convergence).

## 1. Setup and data

In [ ]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path("..").resolve()
if Path("/kaggle").exists() and not REPO.exists():
    !git clone https://github.com/<user>/<repo>.git {REPO}
    !pip install -q faster-coco-eval ultralytics==8.3.*
sys.path.insert(0, str(REPO / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches, cv2
import vdet, parallel, ultra
from vdet import CLASSES

vdet.seed_everything()
ROOT = vdet.find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else REPO / "outputs"
RUNS = OUT / "runs"; RUNS.mkdir(parents=True, exist_ok=True)
RESULT_DIRS = [RUNS, REPO / "results"]   # own runs + runs committed by the teammate
P = vdet.PROTOCOL

# restore finished runs from earlier versions of this notebook (attached as Kaggle input)
for src in (Path("/kaggle/input").glob("*/runs/*") if Path("/kaggle/input").exists() else []):
    if (src / "config.json").exists() and not (RUNS / src.name).exists():
        shutil.copytree(src, RUNS / src.name); print("restored", src.name)

In [ ]:
imgs, anns = vdet.load_all(ROOT)
folds = pd.read_csv(OUT / "folds.csv") if (OUT / "folds.csv").exists() else vdet.make_folds(imgs, P["n_folds"])
folds.to_csv(OUT / "folds.csv", index=False)   # the parallel runner reads the split from this file
train_frames, val_frames = vdet.protocol_frames(imgs, folds, P)
train_gt = anns[anns.key.isin(train_frames.key) & (anns.iscrowd == 0)]
print(json.dumps(P))
print(f"train: {len(train_frames)} frames | val: {len(val_frames)} frames / {val_frames.clip_uid.nunique()} clips")

data_yaml = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=P["val_fold"], out_dir=OUT / "yolo_fold0",
                                    step=P["step"], val_step=P["val_step"], mask_ignore=True)

## 2. Number of queries vs scene density

In [ ]:
n_obj = train_gt.groupby("key").size().reindex(train_frames.key, fill_value=0)
q = n_obj.quantile([.5, .9, .99, 1.0])
print("objects per training frame  p50/p90/p99/max:", q.round(0).astype(int).tolist())
for nq in [300, 500]:
    print(f"frames with more objects than {nq} queries: {(n_obj > nq).mean():.2%}")

Each query can output at most one object, so frames with more objects than queries lose recall by construction. If the p99 above exceeds 300, D1 uses 500 queries (cost: more decoder compute, more unmatched queries per image).

## 3. Experiments

In [ ]:
BASE = dict(family="rtdetr", model="rtdetr-l.pt", imgsz=P["imgsz"], epochs=P["epochs"], batch=4, seed=P["seed"],
            train_args=dict(lr0=1e-4, optimizer="AdamW", warmup_epochs=1))
EXPERIMENTS = {
    "detr_d0_base": BASE,
    "detr_d1_q500": {**BASE, "num_queries": 500},
}
ON = "detr_d1_q500"      # set to the better of d0/d1
EXPERIMENTS["detr_d2_cost_giou"] = {**EXPERIMENTS[ON], "cost_gain": {"class": 2, "bbox": 5, "giou": 4}}
EXPERIMENTS["detr_d3_no_dn"] = {**EXPERIMENTS[ON], "num_denoising": 0}
pd.DataFrame(EXPERIMENTS).T

In [ ]:
# Accelerator "GPU T4 x2": two runs train at the same time, one per GPU (quota is per session hour).
# Queue as many runs as fit into the 12 h session; finished runs are skipped. Logs: runs/<name>.log
RUN = ["detr_d0_base", "detr_d1_q500"]
codes = parallel.run_parallel([(n, EXPERIMENTS[n]) for n in RUN], RUNS, folds_csv=OUT / "folds.csv", data_yaml=data_yaml)

## 4. Results

In [ ]:
COLS = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "epochs"]
def show(res, cols=COLS):
    cols = [c for c in cols if c in res]
    num = [c for c in cols if c not in ("ms_per_img", "epochs")]
    return res[cols].style.format("{:.4f}", subset=num).format("{:.1f}", subset=[c for c in ["ms_per_img"] if c in cols])

res = vdet.collect_results(RESULT_DIRS, "detr")
for p in ("frcnn", "retina", "yolo"):
    o = vdet.collect_results(RESULT_DIRS, p)
    if len(o): res = pd.concat([res, o.loc[[o.AP.idxmax()]]])
if len(res): display(show(res))

## 5. Validation mAP over training: RT-DETR vs YOLOv8 with the same budget

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
runs = [r for r in res.index if r.startswith(("detr", "yolo"))] if len(res) else []
for run in runs:
    h = vdet.load_history(RESULT_DIRS, run)
    if h is None: continue
    axes[0].plot(h.epoch, h.AP_ultra, "o-", label=run)
    if "time" in h: axes[1].plot(h["time"] / 60, h.AP_ultra, "o-", label=run)
axes[0].set_xlabel("epoch"); axes[1].set_xlabel("GPU minutes")
for ax in axes: ax.set_ylabel("val mAP50-95 (Ultralytics validator)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "detr_vs_cnn_curves.png", dpi=120); plt.show()

If the RT-DETR curve is still rising at the last epoch, the model has **not converged** within the budget; say so in the report and compare the slope with YOLO's.

## 6. Small objects and dense scenes

In [ ]:
# AP on holdout frames split by density (objects per frame), for the best run of each family
dens = anns[anns.key.isin(val_frames.key) & (anns.iscrowd == 0)].groupby("key").size().reindex(val_frames.key, fill_value=0)
bins = pd.qcut(dens, 3, labels=["sparse", "medium", "dense"], duplicates="drop")
rows = []
for run in res.index if len(res) else []:
    f = next((Path(d) / run / "val_preds.csv.gz" for d in RESULT_DIRS if (Path(d) / run / "val_preds.csv.gz").exists()), None)
    if f is None: continue
    preds = pd.read_csv(f)
    row = {"run": run}
    for b in bins.cat.categories:
        sub = val_frames[val_frames.key.isin(bins.index[bins == b])]
        r = vdet.coco_evaluate(sub, anns, preds, per_class=False)
        row[f"AP {b}"] = r["AP"]; row[f"AP_small {b}"] = r["AP_small"]
    rows.append(row)
if rows: display(pd.DataFrame(rows).set_index("run").style.format("{:.4f}"))
print("objects per frame in each bin:", dens.groupby(bins, observed=False).agg(["min", "max"]).to_dict("index"))

Discuss: does the gap between RT-DETR and YOLO grow from sparse to dense frames (limited queries, one-to-one matching), and on small objects (the encoder works on strides 8–32 only)? The effect of denoising (D3 vs its parent) should be visible mostly in the first epochs of the curve.

## 7. Export and test submission

In [ ]:
BEST = res.drop(index=[i for i in res.index if not i.startswith("detr")]).AP.idxmax() if len(res) else "detr_d0_base"
model = ultra.load(RUNS / BEST)
test_preds = ultra.predict_test(model, ROOT, P["imgsz"])
test_preds.to_csv(OUT / f"test_preds_{BEST}.csv.gz", index=False)
vdet.write_submission(test_preds, ROOT / "test_images.csv", OUT / f"submission_{BEST}.csv")

In [ ]:
# Results for the repo: unzip results.zip from the notebook Output into <repo>/results/ and commit.
vdet.export_results(RUNS, OUT / "results", preds_for=(BEST,))